# デモ1：言葉で分類する（CLIP）

第1回では、37クラスの犬と猫を分類するために、写真とラベルを用意して学習しました。このデモのモデルは、**学習し直さずに、クラスの名前を言葉で渡すだけで分類します**。インターネット上の大量の「画像と説明文の組」で、画像と文の近さを測れるように学習してあるためです。

- モデル：[laion/CLIP-ViT-B-32-laion2B-s34B-b79K](https://huggingface.co/laion/CLIP-ViT-B-32-laion2B-s34B-b79K)（MIT、約1.5億パラメータ。OpenAI の CLIP と同じ形を、公開データセット LAION-2B で学習したもの）
- 中の Transformer：画像を 32×32 のパッチに分けて読む ViT と、文を読む Transformer の2つ。それぞれの出力のベクトルの近さで、画像と文の合い具合を測る

**最初に**：「ファイル」→「ドライブにコピーを保存」で自分のコピーを作り、「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行します。コードを読む必要はありません。何ができて、何が苦手かを見つけるのが目的です。

## 1. 準備

目安：約2分（うち計算 約1分）

In [ ]:
import importlib.util
import subprocess
import sys

# Colab に入っていない、または古いライブラリを入れる
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import data

device = "cuda" if torch.cuda.is_available() else "cpu"
# T4 は bfloat16 に対応していないので、GPU では float16 を使う
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-image-classification", model="laion/CLIP-ViT-B-32-laion2B-s34B-b79K",
                      device=device, dtype=dtype)

## 2. 例の画像で試す

目安：約3分

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる（JPEG か PNG）

# 例の画像（scikit-image に入っている写真。CC0 またはパブリックドメイン）
images = {
    "astronaut": Image.fromarray(data.astronaut()),
    "cat": Image.fromarray(data.chelsea()),
    "coffee": Image.fromarray(data.coffee()),
}
if USE_UPLOAD:
    import io
    from google.colab import files
    for name, content in files.upload().items():
        images[name] = Image.open(io.BytesIO(content)).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, (name, image) in zip(axes[0], images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

クラスの名前（`LABELS`）を英語で書きます。モデルは英語の文で学習しているので、英語のほうが安定します。出てくる数は、各ラベルである確率です。第1回の分類ヘッドと同じく、合い具合のスコアを softmax で合計1にしています。

宇宙飛行士の写真では、"a person" と "a rocket" に確率が分かれます。背景にスペースシャトルの模型が写っているためです。1枚に複数の物が写っていると、確率はそれらに分かれます。

In [ ]:
LABELS = ["a cat", "a dog", "a person", "a cup of coffee", "a rocket", "a flower"]  # ← 変えてみる

for name, image in images.items():
    results = classifier(image, candidate_labels=LABELS)
    print(f"--- {name} ---")
    for r in results[:3]:
        print(f"  {r['score']:.3f}  {r['label']}")

## 3. やってみよう

目安：約10分

- `LABELS` を変えます。猫の写真に `"a tabby cat"`、`"a Persian cat"`、`"a Siamese cat"` を渡すと、品種まで分けられるでしょうか。第1回で学習したモデルと比べてみましょう。
- 色、場所、気分など、物の名前ではないラベル（`"a photo taken at night"`、`"a happy scene"`）も試します。
- 2. の `USE_UPLOAD = True` にして、自分の写真で試します。
- うまくいかない例を探します。文字が写った写真、絵、似た物が2つ写った写真などはどうでしょうか。